# Exploratory Data Analysis

The aim of this notebook is to understand the horse-racing data before building a model to predict race outcomes.

The analysis covers:
- the structure of the merged dataset;
- numerical and categorical feature distributions;
- class balance;
- historical form for horses, jockeys, and trainers;
- correlations and redundant variables; and
- the relationship between individual features and winning.

**Important:** historical form features are calculated using information from previous races *only*, so the current race outcome is not used to construct its own predictors. This is standard in forecasting tasks, such as predicting winning odds in races.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## 2. Load and merge the raw data

There are two raw tables:

- `runs.csv`: horse-level observations for each race
- `races.csv`: race-level information

After merging, each row represents **one horse in one race**.

In [ ]:
runs = pd.read_csv("../data/runs.csv")
races = pd.read_csv("../data/races.csv")

df = runs.merge(races, on = "race_id", how = "left", validate = "many_to_one")

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")

In [ ]:
df_raw = df.copy()  # make a copy of the original, messy data

## 3. Data types and missing values

In [ ]:
df.info()

In [ ]:
missing = (df.isna().sum()
      .sort_values(ascending=False)
      .to_frame("missing")
      )

missing["missing_pct"] = 100 * missing["missing"] / len(df)

missing[missing["missing"] > 0]

## 4. Parse dates and establish chronological order

The historical form features later in the notebook depend on the order in which races occurred. Parse the date and sort chronologically **before** calculating any prior-performance variables.

In [ ]:
df["date"] = pd.to_datetime(df["date"])  # date initially was an object dtype

df = (df.sort_values(["date", "race_id", "horse_id"]).reset_index(drop=True))

df[["date", "race_id", "horse_id"]].head()

## 5. Target distribution

In [ ]:
target_counts = df["won"].value_counts().sort_index()
target_pct = df["won"].value_counts(normalize=True).sort_index().mul(100)

display(pd.DataFrame({
    "count": target_counts,
    "percentage": target_pct.round(2),
}))

In [ ]:
sns.countplot(data=df, x="won")
plt.xlabel("Won")
plt.ylabel("Number of horses")
plt.title("Class Distribution")
plt.tight_layout()
plt.show()

## 6. Numerical feature distributions

Start with the distributions of the numerical variables to identify skewed features, unusual ranges, and potential data-quality issues.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

df[numeric_cols].hist(
    figsize=(16, 16),
    bins=30,
)

plt.tight_layout()
plt.show()

### Key variables

In [ ]:
initial_features = [
    "horse_age",
    "horse_rating",
    "declared_weight",
    "actual_weight",
    "draw",
    "distance",
    "win_odds",
    "place_odds",
    ]

df[initial_features].hist(figsize=(14, 10), bins=30)

plt.tight_layout()
plt.show()

## 7. Categorical variables

These variables describe the race environment and horse characteristics and are candidates for encoding later.

In [ ]:
categorical_cols = [
    "venue",
    "config",
    "surface",
    "going",
    "horse_country"
    ]

df[categorical_cols].nunique().sort_values().to_frame("n_unique")

### Venue

In [ ]:
venue_races = (df.groupby("venue")["race_id"].nunique().sort_values())

venue_races.plot(kind="bar", figsize=(8, 4))
plt.xlabel("Venue")
plt.ylabel("Number of races")
plt.title("Number of Races by Venue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Horse country

In [ ]:
df["horse_country"].value_counts().sort_values().plot(kind="barh",figsize=(8, 6))

plt.xlabel("Number of horses")
plt.ylabel("Country")
plt.title("Horse Country Distribution")
plt.tight_layout()
plt.show()

### Surface and going

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["surface"].value_counts().plot(kind="bar", ax=axes[0])
axes[0].set_title("Race Surface")
axes[0].set_xlabel("Surface")
axes[0].set_ylabel("Number of horses")

df["going"].value_counts().plot(kind="bar", ax=axes[1])
axes[1].set_title("Going")
axes[1].set_xlabel("Going")
axes[1].set_ylabel("Number of horses")

plt.tight_layout()
plt.show()

In [ ]:
going_stats = (df.groupby("going")["won"].agg(count="count", win_rate="mean"))

going_stats["win_rate"] *= 100

going_stats.sort_values("win_rate", ascending=False)

In [ ]:
going_stats["win_rate"].sort_values().plot(kind="barh",figsize=(8, 5))

plt.xlabel("Win rate (%)")
plt.ylabel("Going")
plt.title("Win Rate by Going")
plt.tight_layout()
plt.show()

## 8. Point-in-time form features

For each horse, jockey, and trainer, calculate:
- number of prior runs;
- number of prior wins; and
- prior win rate.

The current race is excluded from these calculations using `shift()`.

In [ ]:
def add_prior_form_features(df, entity_col, prefix):
    # Number of races before the current race
    df[f"{prefix}_prior_runs"] = df.groupby(entity_col).cumcount()

    # Wins before the current race
    df[f"{prefix}_prior_wins"] = (
        df.groupby(entity_col)["won"]
          .transform(lambda x: x.shift().cumsum())
          .fillna(0)
    )

    # Historical win rate before the current race
    df[f"{prefix}_prior_win_rate"] = (
        df[f"{prefix}_prior_wins"]
        .div(df[f"{prefix}_prior_runs"])
        .fillna(0)
    )

    return df


df = add_prior_form_features(df, "horse_id", "horse")
df = add_prior_form_features(df, "jockey_id", "jockey")
df = add_prior_form_features(df, "trainer_id", "trainer")

### Sanity checks

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(df["horse_prior_runs"],df["horse_prior_win_rate"],alpha=0.2)

plt.xlabel("Horse prior runs")
plt.ylabel("Horse prior win rate")
plt.title("Horse Prior Win Rate vs. Prior Runs")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(df["jockey_prior_runs"],df["jockey_prior_win_rate"],alpha=0.2)
plt.xlabel("Jockey prior runs")
plt.ylabel("Jockey prior win rate")
plt.title("Jockey Prior Win Rate vs. Prior Runs")
plt.tight_layout()
plt.show()

## 9. Correlation analysis

Correlation is useful for identifying variables that contain similar information and potential multicollinearity. It is only a first diagnostic: low linear correlation does not necessarily mean that two variables are unrelated.

In [ ]:
corr = df.corr(numeric_only=True)
display(corr)

In [ ]:
plt.figure(figsize=(18, 14))

sns.heatmap(corr,map="coolwarm",center=0)

plt.title("Numeric Feature Correlation Matrix")
plt.tight_layout()
plt.show()

### Example: horse rating vs. actual weight

In [ ]:
from scipy.stats import pearsonr

r, p = pearsonr(df["horse_rating"],df["actual_weight"])

print(f"Pearson r = {r:.3f}")
print(f"p-value   = {p:.3e}")

## 10. Win rate versus individual features

Comparing observed win rates across feature values gives a simple first indication of whether a variable may be informative.

These plots are **descriptive only** and do not account for confounding variables or differences in field size.

In [ ]:
def plot_win_rate(df, col, target="won"):
    win_rate = (
        df.groupby(col)[target]
          .mean()
          .mul(100)
          .sort_index()
    )

    plt.figure(figsize=(8, 5))
    
    win_rate.plot(marker="o")

    plt.xlabel(col)
    plt.ylabel("Win rate (%)")
    plt.title(f"Win Rate vs. {col}")
    plt.tight_layout()
    plt.show()

    return win_rate

In [ ]:
for feature in ["horse_age", "actual_weight", "draw", "prize"]:
    print(f"\n{feature}\n")
    display(plot_win_rate(df, feature))

## 11. Horse sex

The raw `horse_type` field contains both sex categories and some values that appear to describe coat colour rather than sex. We therefore create a simpler `horse_sex` variable without guessing the meaning of ambiguous categories.

In [ ]:
df["horse_type"].value_counts(dropna=False)

In [ ]:
df["horse_sex"] = df["horse_type"].map({
    "Gelding": "Male",
    "Colt": "Male",
    "Horse": "Male",
    "Rig": "Male",
    "Mare": "Female",
    "Filly": "Female"
    }).fillna("Unknown")

df["horse_sex"].value_counts()

## Summary

The EDA produces:

- a chronologically ordered horse-race dataset;
- an explicit binary target (`won`);
- an overview of numerical and categorical variables;
- point-in-time historical form features for horses, jockeys, and trainers;
- a reduced set of redundant time variables; and
- initial descriptive relationships between features and winning.

The next step should be feature selection/encoding and a **time-aware train/validation split**, so future races are not used to predict earlier races.